In [7]:
from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, input_guardrail, GuardrailFunctionOutput, trace

In [4]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent(
    name="Name Check",
    instructions="Check if user is incuding someone's personal name in what they want you to do.",
    model="gpt-4o-mini",
    output_type=NameCheckOutput
)

In [6]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(starting_agent=guardrail_agent, input=message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info={"found_name": result.final_output.name},
        tripwire_triggered=result.final_output.is_name_in_message
    )

In [9]:
careful_sales_instructions = """You are a humorous, engaging sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write witty, engaging cold emails that are likely to get a response."""

careful_sales_agent = Agent(
    name="Careful Sales Agent",
    instructions=careful_sales_instructions,
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
)

In [10]:
message = "Write a cold sales email to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(starting_agent=careful_sales_agent, input=message)
    print(result.final_output)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [11]:
message = "Write a cold sales email to Dear CEO"

with trace("Protected Automated SDR"):
    result = await Runner.run(starting_agent=careful_sales_agent, input=message)
    print(result.final_output)

Subject: Let’s Make SOC2 Compliance As Easy As Pie (Or Cake, If That’s Your Jam)

Dear CEO,

I hope this email finds you well and enjoying your favorite caffeinated beverage! ☕️ Let’s cut to the chase: compliance isn’t the most thrilling topic, but it’s essential—like that mysterious room in your house that holds all your holiday decorations. 

Enter ComplAI, your soon-to-be favorite SaaS tool! We specialize in turning SOC2 compliance, a process that usually causes more sweat and tears than a marathon, into something that feels like a walk in the park (complete with a refreshing lemonade!). 🍋

What’s our secret sauce, you ask? Well, it’s simple: we use the magic of AI to help you effortlessly prepare for audits, manage documentation, and keep the compliance goblins at bay. You can finally say goodbye to endless spreadsheets and hello to peace of mind! 

*Imagine*: You, sitting back with your feet up, as ComplAI handles the grunt work. Sounds dreamy, right? 

If you’re open to chatting,